In [ ]:
!nvidia-smi

Mon Oct 26 04:28:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls "/content/drive/MyDrive/finsent_data"

finbert_zeroshot_val.csv  tfns_clean.csv


In [ ]:
!git clone -b dev https://github.com/amitojbirah/finsent.git

Cloning into 'finsent'...
remote: Enumerating objects: 85, done.
remote: Counting objects: 100% (85/85), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 85 (delta 21), reused 73 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (85/85), 82.31 KiB | 5.14 MiB/s, done.
Resolving deltas: 100% (21/21), done.


In [ ]:
%cd finsent

/content/finsent


In [ ]:
!mkdir -p data/processed
!cp /content/drive/MyDrive/finsent_data/tfns_clean.csv data/processed/

In [ ]:
!pip install -e . --no-deps

Obtaining file:///content/finsent
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for finsent (pyproject.toml) ... done
  Created wheel for finsent: filename=finsent-0.1.0-0.editable-py3-none-any.whl size=1303 sha256=fad7d613db15d1bd0aec54209cca31429c8d6416ad40b0f0e4afbf01d0fa5546
  Stored in directory: /tmp/pip-ephem-wheel-cache-uuwvqvgx/wheels/06/03/e2/5e0ecddab0f13b7efea29da177624c8bd276b4be33715ded61
Successfully built finsent


In [ ]:
import torch, transformers, datasets, accelerate, sklearn, pandas, dotenv, matplotlib

for lib in [torch, transformers, datasets, accelerate, sklearn, pandas, matplotlib]:
    print(f"{lib.__name__:<14} {lib.__version__}")
print("CUDA available:", torch.cuda.is_available())

from finsent.data.split import load_splits
train, val, test = load_splits()
print(len(train), len(val), len(test))
print(val["label"].value_counts())
print(val["text"].iloc[0])

torch          2.11.0+cu130
transformers   5.20.2
datasets       4.8.5
accelerate     1.15.0
sklearn        1.6.1
pandas         2.2.3
matplotlib     3.10.0
CUDA available: True
7989 1410 2376
label
neutral     911
positive    286
negative    213
Name: count, dtype: int64
$FTS: Fortis announced the appointment of David Hutchens as Chief Operating Officer, Fortis, effective January 1,...


In [ ]:
from transformers import AutoTokenizer , AutoModelForSequenceClassification, pipeline
from sklearn.metrics import f1_score
from finsent.data.split import load_splits



In [ ]:
tokenizer=AutoTokenizer.from_pretrained("ProsusAi/finbert")
model=AutoModelForSequenceClassification.from_pretrained("ProsusAi/finbert")

config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
model.classifier

Linear(in_features=768, out_features=3, bias=True)

In [ ]:
label2id = model.config.label2id
print(label2id)

for name, df in [("train", train), ("val", val), ("test", test)]:
    df["labels"] = df["label"].map(label2id)
    n_missing = df["labels"].isna().sum()
    assert n_missing == 0, f"{name}: {n_missing} labels not found in label2id"
    df["labels"] = df["labels"].astype(int)

print(val["label"].value_counts())
print(val["labels"].value_counts())

{'positive': 0, 'negative': 1, 'neutral': 2}
label
neutral     911
positive    286
negative    213
Name: count, dtype: int64
labels
2    911
0    286
1    213
Name: count, dtype: int64


In [ ]:
import pandas as pd
from finsent.models.evaluate import evaluate

clf = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0)
outputs = clf(val["text"].tolist(), batch_size=64, truncation=True, max_length=64)
zs_preds = [o["label"] for o in outputs]

zs_metrics = evaluate(val["label"], zs_preds)
print(zs_metrics)

zs_df = pd.DataFrame({"text": val["text"], "label": val["label"], "pred": zs_preds})
zs_df.to_csv("/content/drive/MyDrive/finsent_data/finbert_zeroshot_val.csv", index=False)

{'accuracy': 0.7156028368794326, 'macro_f1': 0.6635112194094822, 'f1_negative': 0.603112840466926, 'f1_neutral': 0.7911059098888239, 'f1_positive': 0.5963149078726968}


In [ ]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

MAX_LENGTH = 64


def to_dataset(df):
    """Keep only what Trainer needs and convert pandas -> Hugging Face Dataset."""
    return Dataset.from_pandas(df[["text", "labels"]], preserve_index=False)


def tokenize(batch):
    """Turn a batch of tweets into token IDs (no padding here)."""
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)


train_ds = to_dataset(train).map(tokenize, batched=True, remove_columns=["text"])
val_ds = to_dataset(val).map(tokenize, batched=True, remove_columns=["text"])
test_ds = to_dataset(test).map(tokenize, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Checks
print(train_ds)
print(val_ds.num_rows, test_ds.num_rows)
print(tokenizer.decode(train_ds[0]["input_ids"]))

lengths = [len(ids) for ids in train_ds["input_ids"]]
print("max length:", max(lengths), "| hit the 64 cap:", sum(l == MAX_LENGTH for l in lengths))

Map:   0%|          | 0/7989 [00:00<?, ? examples/s]

Map:   0%|          | 0/1410 [00:00<?, ? examples/s]

Map:   0%|          | 0/2376 [00:00<?, ? examples/s]

Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 7989
})
1410 2376
[CLS] hain celestial eps beats by $ 0. 01, misses on revenue [SEP]
max length: 64 | hit the 64 cap: 4


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments


def compute_metrics(eval_pred):
    """Called by Trainer after each val pass: logits -> predicted IDs -> metrics."""
    logits = eval_pred.predictions
    label_ids = eval_pred.label_ids
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(label_ids, preds),
        "macro_f1": f1_score(label_ids, preds, average="macro", zero_division=0),
    }


training_args = TrainingArguments(
    output_dir="/content/finbert_out",
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=True,
    logging_steps=50,
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

train_result = trainer.train()

print(trainer.evaluate())
print("Training time (s):", round(train_result.metrics["train_runtime"], 1))

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.414348,0.375773,0.860993,0.825346
2,0.258671,0.350935,0.882270,0.846759
3,0.139236,0.411841,0.878014,0.848935
4,0.097022,0.447352,0.873050,0.840146


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.097022,0.411841,4,0.878014,0.848935


{'eval_loss': 0.41184085607528687, 'eval_accuracy': 0.8780141843971632, 'eval_macro_f1': 0.8489346037884812}
Training time (s): 140.1


In [ ]:
import json
import os

import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import f1_score

OUT_DIR = "/content/drive/MyDrive/finsent_data/finbert_ft"
os.makedirs(OUT_DIR, exist_ok=True)

# 1. Model: the loaded best checkpoint (epoch 3) + config + tokenizer
trainer.save_model(OUT_DIR)

# 2. History, settings, run info (for MLflow in Step 7)
with open(f"{OUT_DIR}/log_history.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

with open(f"{OUT_DIR}/training_args.json", "w") as f:
    json.dump(training_args.to_dict(), f, indent=2, default=str)

run_info = {
    "model_name": "ProsusAI/finbert",
    "max_length": MAX_LENGTH,
    "train_runtime_s": train_result.metrics["train_runtime"],
    "best_model_checkpoint": trainer.state.best_model_checkpoint,
    "best_val_macro_f1": trainer.state.best_metric,
    "n_train": len(train_ds),
    "n_val": len(val_ds),
}
with open(f"{OUT_DIR}/run_info.json", "w") as f:
    json.dump(run_info, f, indent=2)

# 3. Predictions -> CSVs with label names and confidence
id2label = model.config.id2label


def save_predictions(ds, df, filename):
    logits = trainer.predict(ds).predictions
    probs = softmax(logits, axis=1)
    pred_ids = probs.argmax(axis=1)
    out = pd.DataFrame(
        {
            "text": df["text"].values,
            "label": df["label"].values,
            "pred": [id2label[int(i)] for i in pred_ids],
            "confidence": probs.max(axis=1),
        }
    )
    out.to_csv(f"{OUT_DIR}/{filename}", index=False)
    return out


val_out = save_predictions(val_ds, val, "val_preds.csv")
# Drop labels so Trainer can't compute test metrics: test stays unscored for now
test_out = save_predictions(test_ds.remove_columns(["labels"]), test, "test_preds.csv")

# 4. Checks
print(len(val_out), len(test_out))
print("val macro-F1 from CSV:", round(f1_score(val_out["label"], val_out["pred"], average="macro"), 4))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

1410 2376
val macro-F1 from CSV: 0.8489
